# Table of Contents

0. [Study Setup & Learning Goals](#Study-Setup-&-Learning-Goals)
1. [Import the Libraries / Load the Data / Housekeeping](#1.-Import-The-Libraries-/-Load-the-Data-/-Housekeeping)
2. [Define Helpful Functions](#2.-Define-Helpful-Functions)
3. [First Look: Inspect Before Testing](#3.-First-Look:-Inspect-Before-Testing)
4. [Data Cleaning](#4.-Data-Cleaning)
5. [Inspect Again After Cleaning](#5.-Inspect-Again-After-Cleaning)
6. [Descriptive Statistics](#6.-Descriptive-Statistics)
   - [Average Time Needed](#Average-Time-Needed)
   - [Visualize Demographics](#Visualize-Demographics-(before-and-after-cleaning))
   - [Test for Gender Differences](#Test-for-Gender-Differences)
7. [Statistical Analysis (finally!)](#7.-Statistical-Analysis-(finally!))
   - [First: Trust](#First:-Trust)
   - [Second: Response Time](#Second:-Response-Time)
8. [How to Report the Results](#8.-How-to-Report-the-Results)


# Study Setup & Learning Goals

This hands-on notebook continues the **running toy user study** from the tutorial.

### Study setup
- **60 simulated participants** (according to an a-priori poweranalysis for 2 independent groups, with desired power = 0.8, α = 0.05 at a large expected effect size d = 0.2)
- **Two between-participant conditions:**  
  - `No-XP`: AI decision without an explanation  
  - `CF-XP`: AI decision accompanied by a counterfactual explanation
- **20 hiring cases (materials) per participant**
- **Primary outcome:** reported **trust** in the AI system, rated from 1 to 5
- **Secondary outcome:** **response time**, measured in seconds

### Running research question
> **How do counterfactual explanations for AI-generated hiring decisions affect users' trust in the AI system?**

For the tutorial, we use the simple hypothesis that participants in the `CF-XP` condition will report higher trust than participants in the `No-XP` condition. We also inspect response time as a secondary outcome, because additional explanation information may require additional processing.

### What this notebook is meant to teach
The data are simulated and deliberately contain a few imperfect response patterns. The goal is **not** to produce a substantive empirical finding. Instead, we practice the workflow from the preceding tutorial section:

**inspect → clean using predefined criteria → inspect again → describe → test → interpret**

> **Tutorial simplification:** We keep the study design and statistical tests intentionally simple. Repeated trial-level observations are aggregated to the participant level before the main inferential tests. In a full study, more complex designs may call for models that explicitly represent repeated observations and variability across both participants and materials.


# 1. Import the Libraries / Load the Data / Housekeeping

First, we load the two CSV files into pandas DataFrames:

- `trust_data`: one trust rating per participant × material
- `reaction_times_task`: one response time per participant × material

The files retain their original `AAAI` filenames for compatibility with the existing tutorial repository, but the same simulated data are used here for the ICPR hands-on session.

We also define a consistent plotting palette so that `No-XP` and `CF-XP` are represented consistently throughout the notebook.


In [ ]:
%pip install pandas numpy matplotlib seaborn scipy scikit_posthocs statsmodels rpy2


In [ ]:
# Import the libraries used throughout the tutorial
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Load the simulated trial-level response-time data (ReactionTime is in seconds)
reaction_times_task = pd.read_csv('https://raw.githubusercontent.com/andreArtelt/ICPR26_CF_Tutorial/main/reaction_time_data_ICPR.csv')

# Load the simulated trial-level trust ratings (1 = low trust, 5 = high trust)
trust_data = pd.read_csv('https://raw.githubusercontent.com/andreArtelt/ICPR26_CF_Tutorial/main/trust_data_ICPR.csv')

# Keep condition colors consistent across all plots
color_palette = {
    'No-XP': '#1f77b4',  # blue
    'CF-XP': '#ff7f0e',  # orange
}

# Convert the palette to a list of colors
colors = [color_palette[key] for key in color_palette]


In [ ]:
print(trust_data)
print(reaction_times_task)


# 2. Define Helpful Functions

These small helper functions keep the plotting and the deliberately simple tutorial cleaning rules readable.

The important methodological point is not the functions themselves, but that **quality-control rules should be defined consistently rather than invented after seeing which observations help or hurt the hypothesis**.


In [ ]:
# Helpers for plotting / initial data inspection
def get_ylim(data, group_column, value_column):
    grouped = data.groupby(group_column)[value_column].agg(['mean', 'sem']).reset_index()
    max_ylim = (grouped['mean'] + 1.5*grouped['sem']).max()
    return 0, max_ylim

def get_material_ylim(data, group_column, material_column, value_column):
    grouped = data.groupby([group_column, material_column])[value_column].agg(['mean', 'sem']).reset_index()
    max_ylim = (grouped['mean'] + 1.2*grouped['sem']).max()
    return 0, max_ylim

# Helper for the predefined population-level response-time cleaning rule
def identify_outliers(df, column):
    # Tutorial rule: flag observations more than 3 SD below the overall mean
    removal_factor_population=3
    print("\nMeasure: " + column)
    print("Population mean: " + str(round(df[column].mean(),3)))
    print("Population std: " + str(round(df[column].std(),3)))
    print("Admissible range: " + str(round(df[column].mean()-removal_factor_population * df[column].std())) + " -- " + str(round(df[column].mean()+removal_factor_population * df[column].std())))
    #outliers = df[np.abs(df[column] - df[column].mean()) > (3 * df[column].std())]
    outliers = df[df[column] < (df[column].mean() - removal_factor_population * df[column].std())]
    return outliers['Participant'].unique()


# 3. First Look: Inspect Before Testing

Before running any inferential test, **look at the data**.

At this stage we are not deciding which observations to remove based on whether they support the hypothesis. We are checking whether the data behave as expected and whether our **predefined quality-control rules** identify anything unusual.

We will look at:
- the composition of the two conditions,
- the overall trust pattern,
- variability between participants,
- variability between materials, and
- response-time patterns.

A useful reminder from the preceding tutorial section: human-study data vary on **both sides of the screen**. Participants differ, but materials can differ too.


In [ ]:
# Start broad: condition composition, group-level patterns, participant-level patterns, and material-level patterns
unique_participants = trust_data.drop_duplicates(subset='Participant')

# Plot Gender distribution
print('\n Plot Gender distribution')
plt.figure(figsize=(8, 4))
sns.countplot(data=unique_participants, x='Gender', hue='Group', palette=color_palette)
plt.title('Gender Distribution by Condition')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Look at Trust
#plt.figure(figsize=(8, 4))
print('\n Plot Trust Values')
plt.figure(figsize=(8, 4))
# Compute and plot mean trust per group
mean_trust_per_user = trust_data.groupby(['Participant','Group'])['Trust'].mean().reset_index()
sns.barplot(data=mean_trust_per_user, x='Group', y='Trust', palette=color_palette, errorbar='se')
plt.title('Mean Trust per User by Condition')
plt.xlabel('Condition')
plt.ylabel('Mean Trust')
plt.ylim(get_ylim(mean_trust_per_user, 'Group', 'Trust'))
plt.tight_layout()
plt.show()

# A closer look: what about individual participants?
# Plot Trust per Participant
plt.figure(figsize=(8, 4))
sns.barplot(x='Participant', y='Trust', hue='Group', data=trust_data, palette=color_palette, errorbar='se')
plt.title('Mean Trust per Participant')
plt.xlabel('Participants')
plt.ylabel('Trust')
plt.ylim(0,max(trust_data['Trust'])+.2)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()

# Look at Reaction Times
# get ylims
max_ylim_times_mean=get_ylim(reaction_times_task, 'Group', 'ReactionTime')
max_ylim_times=get_material_ylim(reaction_times_task, 'Group', 'Material', 'ReactionTime')
max_ylim_trust=get_material_ylim(trust_data, 'Group', 'Material','Trust',)

plt.figure(figsize=(8, 4))
# Plot Reaction Times per Group
print('\n Plot Reaction Times')
sns.barplot(data=reaction_times_task, x='Group', y='ReactionTime', palette=color_palette, errorbar='se')
plt.title('Mean Response Time by Condition')
plt.xlabel('Condition')
plt.ylabel('Response Time (seconds)')
plt.ylim(max_ylim_times_mean)

# Inspect response time by material: materials themselves can be unusually easy / difficult
plt.figure(figsize=(8, 4))
sns.barplot(x='Material', y='ReactionTime', hue='Group', data=reaction_times_task, palette=color_palette, errorbar='se')
plt.title('Mean Response Time per Material Number')
plt.xlabel('Material Number')
plt.ylabel('Response Time (seconds)')
plt.ylim(max_ylim_times)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()

# A closer look: what about individual participants?
# Inspect response time by participant: participants can also show unusual response patterns
plt.figure(figsize=(8, 4))
sns.barplot(x='Participant', y='ReactionTime', hue='Group', data=reaction_times_task, palette=color_palette, errorbar='se')
plt.title('Mean Response Time per Participant')
plt.xlabel('Participants')
plt.ylabel('Response Time')
plt.ylim(0,max(reaction_times_task['ReactionTime'])+2)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()


# 4. Data Cleaning

Now we apply the **predefined tutorial quality-control rules**.

The simulated dataset deliberately contains a few response patterns that make these checks visible. For this hands-on session, we keep the original simple rules and apply them identically across conditions.

### Participant-level rules used here

1. **Very fast responding ("speedsters")**  
   A participant is flagged if the response-time data contain an observation more than **3 SD below the overall response-time mean**.

2. **Straightlining / non-varying trust responses**  
   A participant is flagged if their **available trust ratings have zero variance**, i.e., the same rating is repeated throughout the recorded responses.

Once flagged, participants are removed from **both** datasets so that trust and response-time analyses use the same cleaned sample.

> These rules are useful for demonstrating a transparent QC workflow. They are not universal thresholds that should be copied automatically into every user study; appropriate criteria depend on the task, measures, and preregistered analysis plan.


In [ ]:
# QC 1: identify participants with very fast task responses according to the predefined 3-SD rule
outliers_reaction_times_task = set()
for column in ['ReactionTime']:
    outliers_reaction_times_task.update(identify_outliers(reaction_times_task, column))
    
print("Participant IDs of speedsters relative to all participants (task) :" + str(outliers_reaction_times_task))

# QC 2: identify participants whose available trust ratings show no within-participant variation
print("\nStraightlining behavior:")
# A zero variance means that the same trust rating was repeated throughout the available responses
straightliners_task = trust_data.groupby('Participant').filter(lambda x: (x['Trust'].var() == 0))
print("Participants straightlining during task: " + str(straightliners_task['Participant'].unique()))


In [ ]:
# Combine the participant IDs flagged by either predefined QC rule
all_outliers = outliers_reaction_times_task.union(straightliners_task['Participant'].unique())

print("Participants marked as outliers: " + str(all_outliers))
print("N: " + str(len(all_outliers)))


In [ ]:
# Apply the same exclusion list to both outcome datasets
reaction_times_task_cleaned = reaction_times_task[~reaction_times_task['Participant'].isin(all_outliers)]
trust_data_cleaned = trust_data[~trust_data['Participant'].isin(all_outliers)]


#### Optional extension: unusually fast materials?

*For time reasons, this optional material-level diagnostic is not part of the live ICPR hands-on. The code remains here for later exploration.*

Participant QC is only one side of the story. A particular **material** may also behave unusually. Here, the original tutorial code asks whether there are materials that are answered unusually quickly:

- relative to the overall response-time distribution, or
- relative to a participant's own typical response time.

This is useful as a diagnostic because a material may be confusing, trivial, technically broken, or otherwise different from the rest of the set.

> **Caution:** removing materials can change the balance of a carefully constructed stimulus set. A flagged material should therefore be inspected and justified, not automatically deleted.


In [ ]:
# Optional diagnostic: flag unusually fast trial responses relative to each participant's own RT distribution
# Define the factor with which std will be multiplied to find the admissible range
removal_factor_user_wise = 3
wonky_trial_user_percentage = 30

# Identify trials (=materials) where users performed more quickly than > 3SD from their own RT mean
reaction_time_outliers = reaction_times_task_cleaned[
    reaction_times_task_cleaned['ReactionTime'] < (
        reaction_times_task_cleaned.groupby('Participant')['ReactionTime'].transform('mean') - 
        removal_factor_user_wise * reaction_times_task_cleaned.groupby('Participant')['ReactionTime'].transform('std')
    )
]

# Display the outliers for Response Time
print("\nParticipants + Material of speedsters relative to own performance (response time):")
print(reaction_time_outliers[['Participant', 'Material', 'ReactionTime']])

## Calculate the percentage of outliers for each Material(=material)
outlier_percentage = reaction_time_outliers['Material'].value_counts(normalize=True) * 100

# Identify Material(=material) where at least 30% of participants were quicker than > 3 SDs of their own performance
wonky_trials = outlier_percentage[outlier_percentage >= wonky_trial_user_percentage].index.tolist()

print("\nList of Material where at least 30% of participants were quicker than > 3 SDs of their own performance:")
print(wonky_trials)

## Show specifics of wonky_trials (=material)
wonky_trials_df = trust_data_cleaned[trust_data_cleaned['Material'].isin(wonky_trials)].groupby('Material').head(1)
print("\nSpecifics of wonky materials:")
print(wonky_trials_df)

# If material removal is used, remember that it can disrupt the balance of the material set
def remove_wonky_trials(df, wonky_trials):
    return df[~df['Material'].isin(wonky_trials)]
# Remove wonky trials from all relevant dataframes
reaction_times_task_cleaned = remove_wonky_trials(reaction_times_task_cleaned, wonky_trials)
trust_data_cleaned = remove_wonky_trials(trust_data_cleaned, wonky_trials)


# 5. Inspect Again After Cleaning

Cleaning should be **transparent and inspectable**, not a black box.

We therefore repeat the key visualizations after applying the QC rules. Ask:

- Did the suspicious participant-level patterns disappear?
- Did the balance between conditions change?
- Do the overall trust and response-time patterns still look plausible?
- Are there still unusual materials or participants worth inspecting?

The goal is to understand what the cleaning step changed **before** moving to inferential statistics.


In [ ]:
# Re-run the same core diagnostics after cleaning
unique_participants_cleaned = trust_data_cleaned.drop_duplicates(subset='Participant')

# Plot Gender distribution
print('\n Plot Gender distribution')
plt.figure(figsize=(8, 4))
sns.countplot(data=unique_participants_cleaned, x='Gender', hue='Group', palette=color_palette)
plt.title('Gender Distribution by Condition')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Look at Trust
print('\n Plot Trust Values')
plt.figure(figsize=(8, 4))
# Compute and plot mean trust per group
mean_trust_per_user = trust_data_cleaned.groupby(['Participant','Group'])['Trust'].mean().reset_index()
sns.barplot(data=mean_trust_per_user, x='Group', y='Trust', palette=color_palette, errorbar='se')
plt.title('Mean Trust per User by Condition')
plt.xlabel('Condition')
plt.ylabel('Mean Trust')
plt.ylim(get_ylim(mean_trust_per_user, 'Group', 'Trust'))
plt.tight_layout()
plt.show()

# A closer look: what about individual participants?
# Plot Trust per Participant
plt.figure(figsize=(8, 4))
sns.barplot(x='Participant', y='Trust', hue='Group', data=trust_data_cleaned, palette=color_palette, errorbar='se')
plt.title('Mean Trust per Participant')
plt.xlabel('Participants')
plt.ylabel('Trust')
plt.ylim(0,max(trust_data_cleaned['Trust'])+.2)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()

# Look at Reaction Times
# get ylims
max_ylim_times_mean=get_ylim(reaction_times_task_cleaned, 'Group', 'ReactionTime')
max_ylim_times=get_material_ylim(reaction_times_task_cleaned, 'Group', 'Material', 'ReactionTime')
max_ylim_trust=get_material_ylim(trust_data_cleaned, 'Group', 'Material','Trust',)

plt.figure(figsize=(8, 4))
# Plot Reaction Times per Group
print('\n Plot Reaction Times')
sns.barplot(data=reaction_times_task_cleaned, x='Group', y='ReactionTime', palette=color_palette, errorbar='se')
plt.title('Mean Response Time by Condition')
plt.xlabel('Condition')
plt.ylabel('Response Time (seconds)')
plt.ylim(max_ylim_times_mean)

# Re-check response time by material
plt.figure(figsize=(8, 4))
sns.barplot(x='Material', y='ReactionTime', hue='Group', data=reaction_times_task_cleaned, palette=color_palette, errorbar='se')
plt.title('Mean Response Time per Material Number')
plt.xlabel('Material Number')
plt.ylabel('Response Time (seconds)')
plt.ylim(max_ylim_times)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()

# A closer look: what about individual participants?
# Re-check response time by participant
plt.figure(figsize=(8, 4))
sns.barplot(x='Participant', y='ReactionTime', hue='Group', data=reaction_times_task_cleaned, palette=color_palette, errorbar='se')
plt.title('Mean Response Time per Participant')
plt.xlabel('Participants')
plt.ylabel('Response Time')
plt.ylim(0,max(reaction_times_task_cleaned['ReactionTime'])+2)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()


# 6. Descriptive Statistics

Before testing the hypotheses, describe the sample and the basic properties of the data.

Here we summarize:
- the approximate time participants needed for the task,
- the condition sizes after cleaning, and
- the gender distribution before and after cleaning.

This is also a useful check that exclusions have not accidentally created a severely imbalanced sample.

## Average Time Needed


In [ ]:
sums_per_participant = reaction_times_task_cleaned.groupby('Participant')['ReactionTime'].sum()
mean_total_time = sums_per_participant.mean()
sd_total_time = sums_per_participant.std()
print(mean_total_time)
print('Mean +/- SD total time needed per participant:')
print(str(round(mean_total_time/60,2)) + ' minutes +/- '+str(round(sd_total_time/60,2)))


## Visualize Demographics (before and after cleaning)


In [ ]:
# Provide a summary of the number of participants in each group
participant_summary = unique_participants_cleaned.groupby(['Group','Gender']).size().reset_index(name='Number of Participants')

print("\nSummary of the number of participants in each group:")
print(participant_summary)

# Plot: After data cleaning
plt.figure(figsize=(8, 4))

# Plot Gender Distribution again, as a reminder:
print('\n Plot Gender distribution')
plt.subplot(1, 2, 1)
sns.countplot(data=unique_participants, x='Gender', hue='Group', palette=color_palette)
plt.title('Before Cleaning: Gender Distribution')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(loc='upper right')

plt.subplot(1, 2, 2)
sns.countplot(data=unique_participants_cleaned, x='Gender', hue='Group', palette=color_palette)
plt.title('After Cleaning: Gender Distribution')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


## Test for Gender Differences

For the tutorial, we retain the original simple chi-square check of whether gender distribution differs between the two conditions.

The purpose of this step is not to "prove" that the groups are identical. Rather, we check whether there is statistically reliable evidence of a group difference in this demographic variable before proceeding with the planned simple analysis.


In [ ]:
from scipy.stats import chi2_contingency, chi2, norm, shapiro, levene, kruskal

# Tutorial check: chi-square test of independence for gender distribution across conditions
contingency_table = pd.crosstab(trust_data_cleaned['Group'], trust_data_cleaned['Gender'])
chi2_test_result = chi2_contingency(contingency_table)

print("Chi^2 Test of Independence for Gender Distributions:")
print(f"Chi^2 Statistic: {round(chi2_test_result[0],3)}, p-value: {round(chi2_test_result[1],3)}")
# Interpretation used in this tutorial: p > .05 = no statistically reliable evidence of a gender-distribution difference


**Conclusion:** In this simulated dataset, the chi-square test does not provide statistically reliable evidence that gender distribution differs between the two conditions. We therefore proceed with the planned tutorial analysis without adding gender as a covariate.

*Important:* a non-significant test is not proof that two groups are identical.


# 7. Statistical Analysis (finally!)

Only now do we move from inspection and description to the planned inferential tests.

We examine two outcomes:

1. **Reported trust** in the AI system — our primary outcome in the running example
2. **Response time** — a secondary outcome that can indicate an additional processing-time cost of the explanation

For both analyses, we keep the original tutorial strategy: repeated trial-level observations are first **aggregated to one value per participant**, and the participant is then the unit used in the simple two-group test.

## First: Trust

### Step 1: Plot again to refresh our memory


In [ ]:
## Look at Trust
#plt.figure(figsize=(8, 4))
print('\n Plot Trust Values')
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
# Compute and plot mean trust per group
mean_trust_per_user = trust_data_cleaned.groupby(['Participant','Group'])['Trust'].mean().reset_index()
sns.barplot(data=mean_trust_per_user, x='Group', y='Trust', palette=color_palette, errorbar='se')
plt.title('Mean Trust per User by Condition')
plt.xlabel('Condition')
plt.ylabel('Mean Trust')
plt.ylim(get_ylim(mean_trust_per_user, 'Group', 'Trust'))

# A closer look: what about individual participants?
# Plot Trust per Participant
#plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 2)
sns.barplot(x='Participant', y='Trust', hue='Group', data=trust_data_cleaned, palette=color_palette, errorbar='se')
plt.title('Mean Trust per Participant')
plt.xlabel('Participants')
plt.ylabel('Trust')
plt.ylim(0,max(trust_data_cleaned['Trust'])+.2)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()


### Step 2: Simple two-group case — what test do we use?

For this tutorial, we retain the planned **Mann–Whitney U test** for the participant-level trust summaries.

Why this simple choice here?
- Trust is originally recorded on a 1–5 response scale.
- We compare **two independent participant groups**.
- The Mann–Whitney U test is a non-parametric two-group test with relatively few distributional assumptions.
- Unequal group sizes after predefined exclusions are not a problem for the test.

We first aggregate the 20 trial-level trust ratings to **one mean trust value per participant**, and then compare those participant-level values between `No-XP` and `CF-XP`.

> This is deliberately a teaching example, not a claim that Mann–Whitney U is the only valid analysis for this kind of dataset. In a full repeated-measures study, the analysis should follow the exact design, outcome scale, and inferential target.


In [ ]:
from scipy.stats import mannwhitneyu

# Aggregate repeated trial-level trust ratings to one mean value per participant
participant_trust = trust_data_cleaned.groupby(['Participant', 'Group'])['Trust'].mean().reset_index()
participant_trust_mean = participant_trust.groupby(['Group'])['Trust'].mean().reset_index()
participant_trust_sd = participant_trust.groupby(['Group'])['Trust'].std().reset_index()

print("\nAggregated Trust Data")
print("Means:")
print(participant_trust_mean)
print("Std:")
print(participant_trust_sd)

# Separate participant-level trust summaries by experimental condition
trust_no_xp = participant_trust.loc[participant_trust['Group'] == 'No-XP', 'Trust']
trust_cf_xp = participant_trust.loc[participant_trust['Group'] == 'CF-XP', 'Trust']

# Planned simple two-group comparison: Mann-Whitney U test
stat, p_val = mannwhitneyu(trust_no_xp, trust_cf_xp, alternative='two-sided')

print(f"\nMann-Whitney U statistic: {stat:.3f}, p-value: {p_val:.10f}")


**Conclusion:** In this simulated tutorial dataset, participant-level trust differs between the two conditions, with the `CF-XP` group reporting higher trust than the `No-XP` group.

This supports the **toy hypothesis in this dataset**. It does **not** establish that counterfactual explanations universally improve trust, nor that higher trust is necessarily more appropriate trust.


## Second: Response Time

Next, we analyze whether the two conditions also differ in response time.

This is useful because the explanation condition contains additional information. A longer response time could reflect additional processing demands — but it is not automatically "bad": it could also reflect deeper engagement with the decision and explanation.

### Step 1: Plot again to refresh our memory


In [ ]:
# Plot Reaction Times per Group
print('\n Plot Reaction Times')
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
sns.barplot(data=reaction_times_task_cleaned, x='Group', y='ReactionTime', palette=color_palette, errorbar='se')
plt.title('Mean Response Time by Condition')
plt.xlabel('Condition')
plt.ylabel('Response Time (seconds)')
plt.ylim(max_ylim_times_mean)

# A closer look: what about individual participants?
# Inspect participant-level response-time patterns alongside the group summary
plt.subplot(1, 2, 2)
sns.barplot(x='Participant', y='ReactionTime', hue='Group', data=reaction_times_task_cleaned, palette=color_palette, errorbar='se')
plt.title('Mean Response Time per Participant')
plt.xlabel('Participants')
plt.ylabel('Response Time')
plt.ylim(0,max(reaction_times_task_cleaned['ReactionTime'])+2)
plt.legend(title='Condition')
plt.tight_layout()
plt.show()


### Step 2: Simple two-group case — what test do we use?

Response time is continuous (measured here in **seconds**). For the original tutorial analysis, we use an **independent-samples t-test** on participant-level mean response times.

As with trust, we first aggregate the repeated trial-level observations to **one mean response time per participant**. We then check the assumptions required for the planned t-test:

1. **Independence**
   - This is primarily a design question.
   - After aggregation, each participant contributes one value to one experimental condition.
   - **Our between-participant design provides independent participant-level observations.**

2. **Approximate normality**
   - We inspect whether participant-level mean response times are approximately normally distributed within each condition.
   - In this tutorial, we use the **Shapiro–Wilk test**.

3. **Homogeneity of variances**
   - We check whether the two groups have roughly equal variances.
   - In this tutorial, we use **Levene's test**.

If these checks are satisfactory, we proceed with the planned independent-samples t-test.


In [ ]:
from scipy import stats

# Aggregate repeated trial-level response times to one mean value per participant
participant_rt = reaction_times_task_cleaned.groupby(['Participant', 'Group'])['ReactionTime'].mean().reset_index()
participant_rt_mean = participant_rt.groupby(['Group'])['ReactionTime'].mean().reset_index()
participant_rt_sd = participant_rt.groupby(['Group'])['ReactionTime'].std().reset_index()

print("\nAggregated Response Time Data")
print("Means:")
print(participant_rt_mean)
print("Std:")
print(participant_rt_sd)

# Separate participant-level mean response times by experimental condition
rt_no_xp = participant_rt.loc[participant_rt['Group'] == 'No-XP', 'ReactionTime']
rt_cf_xp = participant_rt.loc[participant_rt['Group'] == 'CF-XP', 'ReactionTime']

### 1. Normality checks for participant-level mean response times
print("\nPerform Shapiro-Wilk Tests to Check Normality")
print("\nInterpretation: p > 0.05 indicates that the data do not significantly deviate from a normal distribution.")
# Shapiro-Wilk Test for No-XP group
shapiro_no_xp = stats.shapiro(rt_no_xp)
print(f"Shapiro-Wilk (No-XP): W={shapiro_no_xp.statistic:.3f}, p={shapiro_no_xp.pvalue:.3f}")

# Shapiro-Wilk Test for CF-XP group
shapiro_cf_xp = stats.shapiro(rt_cf_xp)
print(f"Shapiro-Wilk (CF-XP): W={shapiro_cf_xp.statistic:.3f}, p={shapiro_cf_xp.pvalue:.3f}")

### 2. Homogeneity-of-variance check

# Levene's Test for equal variances
print("\nPerform Levene's Test to Check for Equal Variances")
print("\nInterpretation: p > 0.05 indicates that the group variances can be assumed equal.")
levene_test = stats.levene(rt_no_xp, rt_cf_xp)
print(f"Levene's Test: W={levene_test.statistic:.3f}, p={levene_test.pvalue:.3f}")


**Conclusion:** For this simulated dataset, the planned checks do not indicate problematic violations of independence, approximate normality, or homogeneity of variances at the participant level. We therefore proceed with the planned independent-samples t-test.


In [ ]:
# Perform the independent samples t-test
t_stat, p_val = stats.ttest_ind(rt_no_xp, rt_cf_xp, equal_var=True)  # Use equal_var=False for Welch's t-test if variances differ

t_df = len(rt_no_xp) + len(rt_cf_xp) - 2

print(f"t-statistic: {t_stat:.3f}, p-value: {p_val:.25f}")
print(f"Degrees of freedom: ",t_df) 


**Conclusion:** In this simulated tutorial dataset, participant-level response times differ between conditions, with the `CF-XP` group taking longer than the `No-XP` group.

This is consistent with the idea that presenting an explanation introduces additional processing time. However, **longer response time is not automatically worse usability**: it can reflect cognitive load, additional information processing, or greater engagement.


# 8. How to Report the Results

The final step is to translate the analysis into a concise, transparent result statement.

Because these data are **simulated for teaching purposes**, the text below is an example of how the corresponding analysis could be reported. The important ingredients are:

- state the starting and final sample sizes,
- report and justify exclusions,
- describe the analyzed sample,
- report descriptive statistics alongside inferential tests, and
- interpret the results **within the scope of the study**, without turning "significant" into "universally better".


## Example Results Write-Up

The simulated dataset contained 60 participants. Data from one participant were excluded because the predefined response-time criterion flagged very fast responding. Three additional participants were excluded because their recorded trust responses showed no within-participant variation, meeting the predefined straightlining criterion.

The final analysis therefore included 56 participants assigned to two conditions: 29 participants in `CF-XP` (14 male / 15 female) and 27 participants in `No-XP` (13 male / 14 female). We did not observe statistically reliable evidence of a difference in gender distribution between conditions. On average, participants needed 5.74 minutes (*SD* = 0.48) to complete the task.

For reported trust, participants in the `CF-XP` condition (*M* = 4.20, *SD* = 0.43) had higher participant-level mean trust scores than participants in the `No-XP` condition (*M* = 3.38, *SD* = 0.48), *U* = 87.00, *p* < .001.

Response times also differed between conditions, *t*(54) = 15.12, *p* < .001. Participants in `CF-XP` (*M* = 18.45 s, *SD* = 0.52) took longer on average than participants in `No-XP` (*M* = 15.89 s, *SD* = 0.74).

### Interpretation
For this simulated example, counterfactual explanations are associated with **higher reported trust and longer response times**. That is the result supported by this particular design and dataset.

It would be an overclaim to conclude simply that "counterfactual explanations are better":
- higher trust is not automatically **better or more appropriate trust**;
- longer response time can indicate a processing cost, but also additional engagement;
- the findings are tied to this user group, task, explanation format, and simulated material.

The broader user-study lesson is therefore:

> **Statistical significance answers only part of the human-centered evaluation question.**


# Take-Home Message

A consistent XAI user-study workflow is:

**claim → research question → design → materials → data inspection → predefined cleaning → descriptive statistics → inferential test → careful interpretation**

For this tutorial example:

- `CF-XP` shows higher reported trust than `No-XP`;
- `CF-XP` also takes longer to process;
- neither result, on its own, tells us whether counterfactual explanations are universally "better".

That final interpretation step takes us back to the central question from the tutorial:

> **Good for whom, for what purpose, and in what context?**
